# Week 8: Vector Workflows

This notebook replicates QGIS vector operations using GeoPandas:
- Load and clean data
- Spatial joins
- Calculate density metrics
- Create choropleth maps

In [ ]:
# Setup (run first)
import sys
if 'google.colab' in sys.modules:
    !pip install geopandas contextily mapclassify -q

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("Ready!")

## 1. Load data

Update the paths to match your folder structure.

In [ ]:
DATA = Path("../data/processed/week08")

neighbourhoods = gpd.read_file(DATA / "neighbourhoods.geojson")
incidents = gpd.read_file(DATA / "incidents.geojson")

print(f"Loaded {len(neighbourhoods)} neighbourhoods, {len(incidents)} incidents")
neighbourhoods.head()

## 2. Clean and prepare

Standardise column names and calculate area.

In [ ]:
neighbourhoods = neighbourhoods.rename(columns=str.lower)
incidents = incidents.rename(columns=str.lower)

# Calculate area in km²
neighbourhoods["area_km2"] = neighbourhoods.to_crs(3857).area / 1e6

neighbourhoods[["neighbourhood_id", "area_km2"]].head()

## 3. Spatial join

Count incidents per neighbourhood (like QGIS "Join attributes by location").

In [ ]:
joined = gpd.sjoin(incidents, neighbourhoods, predicate="within", how="left")

counts = joined.groupby("neighbourhood_id").size().rename("incident_count")

neighbourhoods = neighbourhoods.merge(counts, on="neighbourhood_id", how="left")
neighbourhoods["incident_count"] = neighbourhoods["incident_count"].fillna(0)

neighbourhoods[["neighbourhood_id", "incident_count", "area_km2"]].head()

## 4. Calculate rate

Incidents per square kilometre.

In [ ]:
neighbourhoods["rate_per_km2"] = neighbourhoods["incident_count"] / neighbourhoods["area_km2"]

neighbourhoods.sort_values("rate_per_km2", ascending=False).head()

## 5. Map the results

Choropleth map showing incident density.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

neighbourhoods.plot(
    column="rate_per_km2",
    scheme="quantiles",
    k=5,
    cmap="YlOrRd",
    legend=True,
    ax=ax
)

ax.set_title("Incident Rate per km²")
ax.set_axis_off()
plt.show()

## 6. Export

Save for use in QGIS or later notebooks.

In [ ]:
output_path = DATA / "neighbourhoods_summary.gpkg"
neighbourhoods.to_file(output_path, driver="GPKG")
print(f"Saved to {output_path}")

---

**Done!** You've completed a full vector analysis workflow in Python.